# CertVIC -- InternVL2-8B eval on Kaggle (free T4)

Runs **InternVL2-8B** over the CertVIC pilot pairs and writes prediction JSONL in the exact
CertVIC schema, by patching the import-safe provider scaffold and driving it with
`certvic.eval.run_eval` (all leakage / evidence / resume / manifest logic intact).

Two jobs, identical mechanics:
- **presence / intervention** -- 91 reviewed items (182 generations)
- **absent-object control** -- 120 items (240 generations)

A single GPU is enough (~420 generations total). **No paid services. Internet OFF.**
Download `pred_internvl_8b_*_merged.jsonl`, then on your Mac run
`scripts/pilot_report_from_raw.py` (final cell). No paper claims; pilot-only.

## Before you run -- add these as Kaggle inputs

1. **Accelerator:** GPU **T4 x2** (only GPU0 is used) or any single T4/P100. **Internet: OFF.**
2. **certvic package** dataset -- the repo's `certvic/` module (+ `configs/`). Set `CERTVIC_DIR`.
3. **InternVL2-8B weights** dataset -- a *full* HF snapshot of `OpenGVLab/InternVL2-8B`
   **including the `*.py` modeling files** (Internet is OFF and the model uses
   `trust_remote_code`). Create once in a throwaway Internet-ON notebook:
   `from huggingface_hub import snapshot_download; snapshot_download('OpenGVLab/InternVL2-8B', local_dir='internvl2-8b')`
   then *Save Version* and output it as a dataset. Set `WEIGHTS`.
4. **Data bundles** (repo `dist/`), each added as a dataset (Kaggle unzips them):
   - `certvic_main200_session2_data.zip` -> presence. Set `JOBS[0]['input']`.
   - `certvic_absent_object_control.zip`  -> control.  Set `JOBS[1]['input']`.
   Each holds `pilot_eval_tasks_reviewed.jsonl` + images (top-level = edited, `orig/` = original).
5. Deps usually present on Kaggle GPU images: `transformers`, `bitsandbytes`, `einops`, `timm`
   (the preflight cell checks them).

In [ ]:
# ====== EDIT THESE PATHS to match your Kaggle inputs ======
CERTVIC_DIR = "/kaggle/input/certvic"          # dir that contains the 'certvic/' package
WEIGHTS     = "/kaggle/input/internvl2-8b"     # full HF snapshot incl *.py
PROVIDER    = "internvl_8b"
MODEL_ID    = "OpenGVLab/InternVL2-8B"
RUN_TAG     = "main200"
WORK        = "/kaggle/working"

JOBS = [
    {"name": "presence", "input": "/kaggle/input/certvic-session2",
     "out": "pred_internvl_8b_presence_merged.jsonl", "strict_leakage": True},
    {"name": "control",  "input": "/kaggle/input/certvic-absent-control",
     "out": "pred_internvl_8b_control_merged.jsonl",  "strict_leakage": True},
]

In [ ]:
import os, sys, json, importlib
sys.path.insert(0, CERTVIC_DIR)

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
for mod in ("transformers", "bitsandbytes", "einops", "timm"):
    try:
        m = importlib.import_module(mod); print("ok  ", mod, getattr(m, "__version__", ""))
    except Exception as e:
        print("MISSING", mod, "->", e, " (add as dataset or briefly enable internet to pip install)")

import certvic; print("certvic OK from", os.path.dirname(certvic.__file__))

# Self-contained InternVL config so we don't depend on the uploaded bundle's version.
CFG = f"{WORK}/kaggle_internvl.yaml"
open(CFG, "w").write(
    "mode: kaggle_open_vlm\nprovider: internvl_8b\nmodel_id: %s\n"
    "device: cuda\ndtype: bfloat16\nload_in_8bit: true\nbatch_size: 1\n"
    "max_new_tokens: 16\ntemperature: 0.0\npaid_services_enabled: false\n" % MODEL_ID)
print("wrote", CFG)

In [ ]:
# Rewrite each bundle's task image paths to the Kaggle mount.
# Bundle layout: {input}/<file>.jpg = edited, {input}/orig/<file>.jpg = original.
def remap_tasks(job):
    rows = [json.loads(l) for l in open(f"{job['input']}/pilot_eval_tasks_reviewed.jsonl")]
    miss = 0
    for r in rows:
        ob = os.path.basename(r["original_image_path"]); eb = os.path.basename(r["edited_image_path"])
        r["original_image_path"] = f"{job['input']}/orig/{ob}"
        r["edited_image_path"]   = f"{job['input']}/{eb}"
        miss += (not os.path.exists(r["original_image_path"])) + (not os.path.exists(r["edited_image_path"]))
    dst = f"{WORK}/tasks_{job['name']}.jsonl"
    open(dst, "w").writelines(json.dumps(r) + "\n" for r in rows)
    statuses = sorted({r.get("metadata", {}).get("evidence_status") for r in rows})
    print(f"{job['name']}: {len(rows)} tasks -> {dst} | missing images: {miss} | evidence_status={statuses}")
    assert miss == 0, "Some images not found -- check the bundle input path / unzip layout."
    return dst

for job in JOBS:
    job["tasks"] = remap_tasks(job)

In [ ]:
# Load InternVL2-8B once and patch the scaffold's answer().
import torch
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import certvic.providers.open_vlm as ovlm

IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)
_tf = T.Compose([T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
                 T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
def load_image(path):
    # Single 448 tile (InternVL max_num=1) -- sufficient for a yes/no presence question.
    return _tf(Image.open(path).convert("RGB")).unsqueeze(0)

try:
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModel.from_pretrained(WEIGHTS, torch_dtype=torch.bfloat16, quantization_config=qcfg,
                                      trust_remote_code=True, device_map={"": 0}).eval()
except Exception as e:
    print("BitsAndBytesConfig path failed, trying legacy load_in_8bit:", e)
    model = AutoModel.from_pretrained(WEIGHTS, torch_dtype=torch.bfloat16, load_in_8bit=True,
                                      trust_remote_code=True, device_map={"": 0}).eval()
tokenizer = AutoTokenizer.from_pretrained(WEIGHTS, trust_remote_code=True, use_fast=False)
_GEN = dict(max_new_tokens=16, do_sample=False)

@torch.inference_mode()
def _internvl_answer(self, image_path, prompt):
    pv = load_image(image_path).to(torch.bfloat16).cuda()
    return str(model.chat(tokenizer, pv, "<image>\n" + prompt, _GEN)).strip()

ovlm.OpenVLMProvider.load   = lambda self: None
ovlm.OpenVLMProvider.answer = _internvl_answer
print("InternVL2-8B loaded + OpenVLMProvider.answer patched.")

In [ ]:
# SMOKE TEST -- confirm parseable yes/no BEFORE the full run.
from certvic.eval.parse import parse_answer
for r in [json.loads(l) for l in open(JOBS[0]["tasks"])][:2]:
    for variant, key in [("original", "original_image_path"), ("edited", "edited_image_path")]:
        raw = _internvl_answer(None, r[key], r["question_original"])
        p = parse_answer(raw, "yes_no", strict=True)
        print(f"{r['item_id'][:22]:22s} {variant:8s} raw={raw!r:12s} parsed={p.parsed_answer} ok={p.parse_ok}")
print("If answers are yes/no with ok=True, proceed. Else fix the patch (chat format / dtype).")

In [ ]:
# Full run: in-process run_eval, one job at a time (single GPU, num_shards=1).
from certvic.eval.run_eval import run_eval
import time
for job in JOBS:
    out = f"{WORK}/{job['out']}"; t0 = time.time()
    summary = run_eval(config_path=CFG, tasks_path=job["tasks"], out_path=out,
                       provider_name=PROVIDER, run_id=f"{RUN_TAG}_{PROVIDER}_{job['name']}",
                       num_shards=1, strict_leakage=job["strict_leakage"],
                       evidence_run=True, fail_fast=False, overwrite=False)
    print(job["name"], summary, f"({time.time()-t0:.0f}s) -> {out}")

In [ ]:
# Sanity: parse rate + answer distribution + provider stamp per job.
import collections
for job in JOBS:
    rows = [json.loads(l) for l in open(f"{WORK}/{job['out']}")]
    ans = collections.Counter(r["parsed_answer"] for r in rows)
    okr = sum(r["parse_ok"] for r in rows) / len(rows)
    print(f"{job['name']}: {len(rows)} preds | parse_ok={okr:.3f} | answers={dict(ans)} "
          f"| provider={sorted({r['provider_name'] for r in rows})}")

In [ ]:
# Zip predictions + manifests for download.
import glob, zipfile
files = sorted(glob.glob(f"{WORK}/pred_internvl_8b_*_merged.jsonl")
               + glob.glob(f"{WORK}/pred_internvl_8b_*_merged.jsonl.run_manifest.json"))
with zipfile.ZipFile(f"{WORK}/internvl_preds.zip", "w") as z:
    for f in files: z.write(f, os.path.basename(f))
print("wrote internvl_preds.zip ->", [os.path.basename(f) for f in files])
print("Download both pred_internvl_8b_*_merged.jsonl (presence + control).")

## Back on the Mac

```bash
cd /Users/saketmaganti/Projects/certVIC
python3 scripts/pilot_report_from_raw.py \
  --provider internvl_8b --model-name OpenGVLab/InternVL2-8B --run-label internvl_8b \
  --raw-presence /path/to/pred_internvl_8b_presence_merged.jsonl \
  --raw-control  /path/to/pred_internvl_8b_control_merged.jsonl
```

Writes `data/results/main_real_200/pilot_report__internvl_8b/` (+ sha256-locked raw) and
refreshes `multimodel_pilot_summary.{md,csv,json}`. It REFUSES if a file is missing or its
`provider_name` != internvl_8b. Pilot-only; no paper claim.

**LLaVA-OneVision** later: same notebook, swap `WEIGHTS`/`MODEL_ID`
(`lmms-lab/llava-onevision-qwen2-7b-ov`), `PROVIDER=llava_onevision_7b`, and the load/patch
cell for `LlavaOnevisionForConditionalGeneration` + its processor.